[Énoncés : Série 8 — Récursivité, programmation dynamique et backtracking](../serie8_dp_backtracking.ipynb)

# Corrigé — Série 8 — Récursivité, programmation dynamique et backtracking

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 42 — Aplatir des listes imbriquées

**Idée** : la **récursivité**. Pour chaque élément : si c'est une liste, on applique le même traitement à cette sous-liste ; sinon, c'est un entier qu'on ajoute au résultat. Une profondeur de 100 reste loin de la limite de Python (environ 1 000 appels imbriqués). Version sans récursion possible avec une pile explicite.

**Complexité** : O(nombre total d'éléments et de sous-listes).

In [ ]:
def aplatir(elements: list) -> list[int]:
    resultat = []

    def explorer(liste):
        for e in liste:
            if isinstance(e, list):
                explorer(e)              # une sous-liste : on l'explore à son tour (récursion)
            else:
                resultat.append(e)

    explorer(elements)
    return resultat


verifier(aplatir, [
    (([[1, 2], [3, [4]], 5],), [1, 2, 3, 4, 5]),
    (([],), []),
    (([[], [[]]],), []),   # que des listes vides
    (([1, [2, [3, [4, [5]]]]],), [1, 2, 3, 4, 5]),
    (([[[7]], 8, [9, [10]]],), [7, 8, 9, 10]),
], ne_pas_modifier=True)

## Exercice 43 — Chemins dans une grille avec obstacles

**Idée** : la **programmation dynamique**. On n'énumère pas les chemins (il peut y en avoir des milliards) : on **compte**. Pour arriver sur une case libre, le dernier pas vient forcément d'en haut ou de la gauche, donc `chemins[i][j] = chemins[i-1][j] + chemins[i][j-1]`. On remplit le tableau ligne par ligne, chaque case utilisant des cases déjà calculées.

**Complexité** : O(lignes × colonnes).

In [ ]:
def chemins_avec_obstacles(grille: list[list[int]]) -> int:
    lignes, colonnes = len(grille), len(grille[0])
    # chemins[i][j] = nombre de chemins du départ jusqu'à la case (i, j)
    chemins = [[0] * colonnes for _ in range(lignes)]
    for i in range(lignes):
        for j in range(colonnes):
            if grille[i][j] == 1:
                chemins[i][j] = 0                     # obstacle : aucun chemin n'y passe
            elif i == 0 and j == 0:
                chemins[i][j] = 1                     # le départ
            else:
                depuis_haut = chemins[i - 1][j] if i > 0 else 0
                depuis_gauche = chemins[i][j - 1] if j > 0 else 0
                chemins[i][j] = depuis_haut + depuis_gauche
    return chemins[lignes - 1][colonnes - 1]


verifier(chemins_avec_obstacles, [
    (([[0, 0, 0], [0, 1, 0], [0, 0, 0]],), 2),
    (([[0]],), 1),   # départ = arrivée
    (([[1]],), 0),
    (([[0, 1], [1, 0]],), 0),   # arrivée inaccessible
    (([[0, 0], [0, 0]],), 2),
    (([[0, 0, 0], [0, 0, 0], [0, 0, 0]],), 6),
    (([[0, 0, 0, 0]],), 1),
    (([[0, 0], [0, 1]],), 0),   # arrivée bloquée
])

## Exercice 44 — Gains sans choisir deux voisins

**Idée** : programmation dynamique avec deux valeurs qu'on fait avancer : le meilleur gain si l'on **prend** la position courante (alors on n'a pas pris la précédente : `sans + g`), et le meilleur gain si on **ne la prend pas** (`max(avec, sans)` de l'étape d'avant). Pas besoin de tableau : seules les deux dernières valeurs servent.

**Complexité** : O(n), mémoire O(1). Essayer toutes les sélections coûterait O(2ⁿ).

In [ ]:
def gain_sans_voisins(gains: list[int]) -> int:
    # avec = meilleur gain jusqu'ici si l'on PREND la position courante
    # sans = meilleur gain jusqu'ici si l'on NE la prend PAS
    avec, sans = 0, 0
    for g in gains:
        avec, sans = sans + g, max(avec, sans)   # prendre g interdit d'avoir pris la précédente
    return max(avec, sans)


verifier(gain_sans_voisins, [
    (([2, 7, 9, 3, 1],), 12),
    (([-5, -1],), 0),
    (([],), 0),
    (([5],), 5),
    (([1, 2, 3, 1],), 4),
    (([2, 1, 1, 2],), 4),   # on peut sauter deux positions d'affilée
    (([10, -1, -1, 10],), 20),
    (([3, -2, 5, -1, 6],), 14),
])

## Exercice 45 — Nombre de façons de payer

**Idée** : programmation dynamique sur le montant. On ajoute les valeurs de pièces **une par une** : après avoir traité la pièce `p`, `facons[s]` compte les combinaisons qui n'utilisent que les pièces déjà vues. Pour chaque `s`, on ajoute les combinaisons qui finissent par une pièce `p` : `facons[s - p]`. Comme les pièces sont introduites dans un ordre fixe, `1 + 2` et `2 + 1` ne sont comptés qu'une fois. Inverser les deux boucles compterait les **suites** ordonnées (l'exercice 58).

**Complexité** : O(montant × nombre de pièces).

In [ ]:
def combinaisons_monnaie(montant: int, pieces: list[int]) -> int:
    # facons[s] = nombre de combinaisons pour payer s avec les pièces déjà traitées
    facons = [0] * (montant + 1)
    facons[0] = 1                         # payer 0 : une seule façon, ne rien donner
    for piece in pieces:                  # une valeur de pièce après l'autre → l'ordre ne compte pas
        for s in range(piece, montant + 1):
            facons[s] += facons[s - piece]
    return facons[montant]


verifier(combinaisons_monnaie, [
    ((5, [1, 2, 5]), 4),
    ((0, [1, 2]), 1),   # montant 0 : une combinaison (rien)
    ((3, [2]), 0),
    ((10, [10]), 1),
    ((4, [1, 2, 3]), 4),
    ((7, []), 0),   # aucune pièce
    ((100, [1, 5, 10, 25, 50]), 292),
])

In [ ]:
# Test de vitesse
chrono(combinaisons_monnaie, 1_000, [1, 2, 5, 10, 20, 50, 100, 200], attendu=321_335_886)

## Exercice 46 — Codes binaires sans deux 1 consécutifs

**Idée** : le **backtracking** (retour en arrière). On construit le code caractère par caractère ; à chaque étape on essaie `"0"`, puis `"1"` seulement si le dernier caractère n'est pas déjà un `1`. Essayer `"0"` avant `"1"` produit directement l'ordre lexicographique, sans tri. On ne génère jamais de code invalide, au lieu de tout générer puis de filtrer.

**Complexité** : proportionnelle au nombre de codes produits (qui croît comme la suite de Fibonacci) × n.

In [ ]:
def codes_sans_11(n: int) -> list[str]:
    resultat = []

    def construire(code):
        if len(code) == n:
            resultat.append(code)
            return
        construire(code + "0")                 # "0" avant "1" → ordre lexicographique
        if not code.endswith("1"):             # interdit : un "1" juste après un "1"
            construire(code + "1")

    construire("")
    return resultat


verifier(codes_sans_11, [
    ((3,), ['000', '001', '010', '100', '101']),
    ((0,), ['']),   # une seule chaîne : la chaîne vide
    ((1,), ['0', '1']),
    ((2,), ['00', '01', '10']),
    ((4,), ['0000', '0001', '0010', '0100', '0101', '1000', '1001', '1010']),
])

## Exercice 47 — Composer une équipe de k personnes

**Idée** : backtracking. On ajoute une personne, on complète l'équipe récursivement avec des numéros **plus grands** (c'est ce qui garantit l'ordre croissant et évite les doublons comme `[2, 1]`), puis on la retire pour essayer la suivante. Piège classique : ajouter `equipe` au résultat sans copie (`equipe[:]`) ; toutes les équipes du résultat seraient alors la même liste, vide à la fin.

**Complexité** : proportionnelle au nombre d'équipes, C(n, k), × k.

In [ ]:
def equipes_de_k(n: int, k: int) -> list[list[int]]:
    resultat = []
    equipe = []

    def choisir(prochain):
        if len(equipe) == k:
            resultat.append(equipe[:])         # une COPIE : equipe va encore changer
            return
        for p in range(prochain, n + 1):
            equipe.append(p)
            choisir(p + 1)                     # les suivants sont plus grands que p
            equipe.pop()                       # retour en arrière : on retire p et on essaie le suivant

    choisir(1)
    return resultat


verifier(equipes_de_k, [
    ((4, 2), [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]),
    ((3, 0), [[]]),   # k = 0 : l'équipe vide
    ((2, 3), []),   # k > n : impossible
    ((3, 3), [[1, 2, 3]]),
    ((4, 1), [[1], [2], [3], [4]]),
    ((4, 3), [[1, 2, 3], [1, 2, 4], [1, 3, 4], [2, 3, 4]]),
])

## Exercice 48 — Le sac à dos de capacité limitée

**Idée** : programmation dynamique sur la capacité. Pour chaque objet et chaque capacité `c`, deux choix : ne pas le prendre (`meilleur[c]`) ou le prendre (`meilleur[c - p] + v`). La boucle sur `c` va **de droite à gauche** pour que `meilleur[c - p]` corresponde encore à l'état « sans cet objet » ; de gauche à droite, on pourrait prendre le même objet plusieurs fois. Un glouton (meilleur rapport valeur/poids d'abord) ne marche pas ici.

**Complexité** : O(nombre d'objets × capacité).

In [ ]:
def meilleure_valeur_sac(poids: list[int], valeurs: list[int], capacite: int) -> int:
    # meilleur[c] = meilleure valeur pour une capacité c, avec les objets déjà traités
    meilleur = [0] * (capacite + 1)
    for p, v in zip(poids, valeurs):
        for c in range(capacite, p - 1, -1):     # de droite à gauche : chaque objet pris une seule fois
            meilleur[c] = max(meilleur[c], meilleur[c - p] + v)   # sans l'objet, ou avec
    return meilleur[capacite]


verifier(meilleure_valeur_sac, [
    (([2, 3, 4], [4, 5, 6], 5), 9),
    (([], [], 10), 0),
    (([5], [10], 4), 0),   # l'objet est trop lourd
    (([5], [10], 5), 10),
    (([1, 3, 4, 5], [1, 4, 5, 7], 7), 9),
    (([10, 20, 30], [60, 100, 120], 50), 220),   # le glouton par valeur/poids donnerait 160
    (([1, 1, 1], [10, 20, 30], 2), 50),
])

---

[Énoncés de la série](../serie8_dp_backtracking.ipynb)